In [1]:
"""
Factor Analysis with K-Fold Cross Validation
"""

# Google Colab setup
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Modify this path according to your Google Drive structure
    folder_path = "/content/drive/My Drive/Factordata"  # Update this path!
    print("Google Drive mounted successfully")
except:
    print("Not running in Google Colab or drive mount failed")
    folder_path = "."  # Default to current directory if not in Colab

# Keep all existing imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, KFold, cross_validate
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# Add new imports for active learning
from sklearn.base import clone

class ActiveLearner:
    def __init__(self, base_estimator, initial_size=0.1, query_size=10):
        """
        Initialize the active learner.

        Parameters:
        base_estimator: Base classifier to use
        initial_size: Fraction of data to use for initial training
        query_size: Number of samples to query in each iteration
        """
        self.estimator = base_estimator
        self.initial_size = initial_size
        self.query_size = query_size
        self.labeled_indices = None
        self.unlabeled_indices = None
        self.performance_history = []

    def initialize_training(self, X, y):
        """Initialize training with a small random subset."""
        n_samples = len(X)
        n_initial = int(n_samples * self.initial_size)

        # Randomly select initial samples
        all_indices = np.arange(n_samples)
        self.labeled_indices = np.random.choice(all_indices, size=n_initial, replace=False)
        self.unlabeled_indices = np.setdiff1d(all_indices, self.labeled_indices)

        # Train initial model
        self.estimator.fit(X[self.labeled_indices], y[self.labeled_indices])
        initial_score = self.estimator.score(X[self.labeled_indices], y[self.labeled_indices])
        self.performance_history.append(initial_score)

        return initial_score

    def uncertainty_sampling(self, X):
        """Query the most uncertain samples using entropy."""
        probas = self.estimator.predict_proba(X[self.unlabeled_indices])
        uncertainties = -np.sum(probas * np.log2(probas + 1e-10), axis=1)
        return uncertainties

    def query_samples(self, X):
        """Select the next batch of samples to be labeled."""
        uncertainties = self.uncertainty_sampling(X)
        query_indices = np.argsort(uncertainties)[-self.query_size:]
        return self.unlabeled_indices[query_indices]

    def update(self, X, y, new_indices):
        """Update the model with newly labeled samples."""
        # Add new samples to labeled set
        self.labeled_indices = np.concatenate([self.labeled_indices, new_indices])
        self.unlabeled_indices = np.setdiff1d(self.unlabeled_indices, new_indices)

        # Retrain model
        self.estimator.fit(X[self.labeled_indices], y[self.labeled_indices])
        score = self.estimator.score(X[self.labeled_indices], y[self.labeled_indices])
        self.performance_history.append(score)

        return score

def plot_learning_curve(active_learner, save_path, measure):
    """Plot the learning curve of the active learning process."""
    plt.figure(figsize=(10, 6))
    plt.plot(range(len(active_learner.performance_history)),
             active_learner.performance_history,
             marker='o')
    plt.xlabel('Learning Iteration')
    plt.ylabel('Model Performance (Accuracy)')
    plt.title(f'Active Learning Curve - {measure}')
    plt.grid(True)
    plt.savefig(os.path.join(save_path, f'{measure}_active_learning_curve.png'))
    plt.close()

# Modify the main function to include active learning
def main():
    factors = ['Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']
    plots_dir = os.path.join(folder_path, 'analysis_plots')
    os.makedirs(plots_dir, exist_ok=True)
    all_results = []

    for measure in factors:
        print(f"\nProcessing measure: {measure}")
        processed_data = []

        if processed_data:
            # [Previous feature preparation code remains the same until model definition]

            # Initialize active learners for each classifier
            active_learners = {
                'Random Forest': ActiveLearner(
                    RandomForestClassifier(n_estimators=100, random_state=42),
                    initial_size=0.1,
                    query_size=10
                ),
                'Logistic Regression': ActiveLearner(
                    LogisticRegression(max_iter=1000, random_state=42),
                    initial_size=0.1,
                    query_size=10
                ),
                'SVM': ActiveLearner(
                    SVC(probability=True, random_state=42),
                    initial_size=0.1,
                    query_size=10
                ),
                'Gradient Boosting': ActiveLearner(
                    GradientBoostingClassifier(random_state=42),
                    initial_size=0.1,
                    query_size=10
                )
            }

            # Perform active learning for each classifier
            for name, active_learner in active_learners.items():
                print(f"\nPerforming active learning for {name} on {measure}...")

                # Convert to numpy for easier indexing
                X_np = X_scaled.to_numpy()
                y_np = y.to_numpy()

                # Initialize active learning
                initial_score = active_learner.initialize_training(X_np, y_np)
                print(f"Initial score: {initial_score:.4f}")

                # Active learning loop
                n_iterations = 10  # Number of active learning iterations
                for i in range(n_iterations):
                    # Query new samples
                    query_indices = active_learner.query_samples(X_np)

                    # Update model (in real-world scenario, we would get labels here)
                    score = active_learner.update(X_np, y_np, query_indices)
                    print(f"Iteration {i+1}, Score: {score:.4f}")

                # Plot learning curve
                plot_learning_curve(active_learner, plots_dir, f"{measure}_{name}")

                # Store results
                final_results = {
                    'Model': name,
                    'Measure': measure,
                    'Initial_Score': initial_score,
                    'Final_Score': score,
                    'N_Labeled_Samples': len(active_learner.labeled_indices),
                    'Learning_Rate': (score - initial_score) / n_iterations
                }
                all_results.append(final_results)

if __name__ == "__main__":
    main()

Mounted at /content/drive
Google Drive mounted successfully

Processing measure: Alpha..annualisiert.

Processing measure: Value.Growth

Processing measure: Small.Large

Processing measure: Momentum

Processing measure: Volatility


In [2]:


def process_one_measure(df, measure):
    """
    Add rolling statistics for both window sizes.
    """
    for window in [12, 24]:
        df[f'{measure}_Rolling_Mean_{window}'] = df[measure].rolling(window=window, min_periods=1).mean()
        df[f'{measure}_Rolling_Std_{window}'] = df[measure].rolling(window=window, min_periods=1).std()
        df[f'{measure}_Dynamic_Outlier_{window}'] = (
            (df[measure] - df[f'{measure}_Rolling_Mean_{window}']).abs() > 2 * df[f'{measure}_Rolling_Std_{window}']
        )
    return df

def plot_factor_analysis(df, measure, save_path):
    """
    Create and save analysis plots for the given measure.
    """
    plt.figure(figsize=(15, 10))

    plt.subplot(2, 2, 1)
    for source in df['Source_File'].unique():
        fund_data = df[df['Source_File'] == source]
        plt.plot(fund_data.index, fund_data[measure], label=source, alpha=0.4)
        outliers = fund_data[fund_data[f'{measure}_Dynamic_Outlier_12']]
        plt.scatter(outliers.index, outliers[measure], marker='x', alpha=0.6)
    plt.title(f'{measure} Values Over Time (Window=12)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)

    # ... [rest of the plotting code remains the same]
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f'{measure}_analysis.png'), bbox_inches='tight')
    plt.close()

def main():
    factors = ['Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']
    plots_dir = os.path.join(folder_path, 'analysis_plots')
    os.makedirs(plots_dir, exist_ok=True)
    all_results = []

    for measure in factors:
        print(f"\nProcessing measure: {measure}")
        processed_data = []

        # Find Excel files
        excel_files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx") and not f.startswith("Model_Comparison")]

        # Process each file
        for filename in excel_files:
            file_path = os.path.join(folder_path, filename)
            print(f"\nProcessing file: {file_path}")

            try:
                df = pd.read_excel(file_path)
                if measure in df.columns:
                    processed_df = process_one_measure(df.copy(), measure)
                    processed_df['Source_File'] = os.path.basename(file_path).replace('.xlsx', '')
                    processed_data.append(processed_df)
                    print(f"Successfully processed {filename}")
            except Exception as e:
                print(f"Error processing {filename}: {str(e)}")

        if processed_data:
            # Combine all processed data
            combined_data = pd.concat(processed_data)
            combined_data['Date'] = pd.to_datetime(combined_data['Date'])
            combined_data.set_index('Date', inplace=True)

            # Create feature matrix
            features = pd.DataFrame({
                'Value': combined_data[measure],
                'Rolling_Mean_12': combined_data[f'{measure}_Rolling_Mean_12'],
                'Rolling_Std_12': combined_data[f'{measure}_Rolling_Std_12'],
                'Rolling_Mean_24': combined_data[f'{measure}_Rolling_Mean_24'],
                'Rolling_Std_24': combined_data[f'{measure}_Rolling_Std_24'],
                'Fund': combined_data['Source_File'],
                'Anomaly_Label_12': combined_data[f'{measure}_Dynamic_Outlier_12'].astype(int)
            })

            features = features.dropna()

            # Prepare data for modeling
            X = features[['Value', 'Rolling_Mean_12', 'Rolling_Std_12', 'Rolling_Mean_24', 'Rolling_Std_24']]
            y = features['Anomaly_Label_12']

            # Scale features
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)
            X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

            # Plot factor analysis
            plot_factor_analysis(combined_data, measure, plots_dir)

            # Initialize active learners
            active_learners = {
                'Random Forest': ActiveLearner(
                    RandomForestClassifier(n_estimators=100, random_state=42),
                    initial_size=0.1,
                    query_size=10
                ),
                'Logistic Regression': ActiveLearner(
                    LogisticRegression(max_iter=1000, random_state=42),
                    initial_size=0.1,
                    query_size=10
                ),
                'SVM': ActiveLearner(
                    SVC(probability=True, random_state=42),
                    initial_size=0.1,
                    query_size=10
                ),
                'Gradient Boosting': ActiveLearner(
                    GradientBoostingClassifier(random_state=42),
                    initial_size=0.1,
                    query_size=10
                )
            }

            # Perform active learning for each classifier
            for name, active_learner in active_learners.items():
                print(f"\nPerforming active learning for {name} on {measure}...")

                X_np = X_scaled.to_numpy()
                y_np = y.to_numpy()

                initial_score = active_learner.initialize_training(X_np, y_np)
                print(f"Initial score: {initial_score:.4f}")

                for i in range(10):  # 10 active learning iterations
                    query_indices = active_learner.query_samples(X_np)
                    score = active_learner.update(X_np, y_np, query_indices)
                    print(f"Iteration {i+1}, Score: {score:.4f}")

                plot_learning_curve(active_learner, plots_dir, f"{measure}_{name}")

                final_results = {
                    'Model': name,
                    'Measure': measure,
                    'Initial_Score': initial_score,
                    'Final_Score': score,
                    'N_Labeled_Samples': len(active_learner.labeled_indices),
                    'Learning_Rate': (score - initial_score) / 10
                }
                all_results.append(final_results)

            # Save results to Excel
            final_results_df = pd.DataFrame(all_results)
            final_results_path = os.path.join(folder_path, "Active_Learning_Results.xlsx")

            with pd.ExcelWriter(final_results_path, engine='openpyxl') as writer:
                final_results_df.to_excel(writer, sheet_name='Overall_Results', index=False)

                # Create summary sheets
                pivot_measure = final_results_df.pivot_table(
                    index='Model',
                    columns='Measure',
                    values=['Final_Score', 'Learning_Rate'],
                    aggfunc='mean'
                )
                pivot_measure.to_excel(writer, sheet_name='Summary_by_Measure')

                pivot_model = final_results_df.pivot_table(
                    index='Measure',
                    columns='Model',
                    values=['Final_Score', 'Learning_Rate'],
                    aggfunc='mean'
                )
                pivot_model.to_excel(writer, sheet_name='Summary_by_Model')

            print(f"Results saved to {final_results_path}")
        else:
            print(f"No data processed for measure: {measure}")

if __name__ == "__main__":
    main()


Processing measure: Alpha..annualisiert.

Processing file: /content/drive/My Drive/Factordata/zCapital.xlsx
Successfully processed zCapital.xlsx

Processing file: /content/drive/My Drive/Factordata/3645.xlsx
Successfully processed 3645.xlsx

Processing file: /content/drive/My Drive/Factordata/creditsuisse.xlsx
Successfully processed creditsuisse.xlsx

Processing file: /content/drive/My Drive/Factordata/21216.xlsx
Successfully processed 21216.xlsx

Processing file: /content/drive/My Drive/Factordata/IAM.xlsx
Successfully processed IAM.xlsx

Processing file: /content/drive/My Drive/Factordata/GAM.xlsx
Successfully processed GAM.xlsx

Processing file: /content/drive/My Drive/Factordata/Vontobel.xlsx
Successfully processed Vontobel.xlsx

Processing file: /content/drive/My Drive/Factordata/Lo.xlsx
Successfully processed Lo.xlsx

Processing file: /content/drive/My Drive/Factordata/SaraSelect.xlsx
Successfully processed SaraSelect.xlsx

Processing file: /content/drive/My Drive/Factordata/Fin

In [3]:


def process_one_measure(df, measure):
    """
    Add rolling statistics for both window sizes.
    """
    for window in [12, 24]:
        df[f'{measure}_Rolling_Mean_{window}'] = df[measure].rolling(window=window, min_periods=1).mean()
        df[f'{measure}_Rolling_Std_{window}'] = df[measure].rolling(window=window, min_periods=1).std()
        df[f'{measure}_Dynamic_Outlier_{window}'] = (
            (df[measure] - df[f'{measure}_Rolling_Mean_{window}']).abs() > 2 * df[f'{measure}_Rolling_Std_{window}']
        )
    return df

def plot_factor_analysis(df, measure, save_path):
    """
    Create and save analysis plots for the given measure.
    """
    plt.figure(figsize=(15, 10))

    plt.subplot(2, 2, 1)
    for source in df['Source_File'].unique():
        fund_data = df[df['Source_File'] == source]
        plt.plot(fund_data.index, fund_data[measure], label=source, alpha=0.4)
        outliers = fund_data[fund_data[f'{measure}_Dynamic_Outlier_12']]
        plt.scatter(outliers.index, outliers[measure], marker='x', alpha=0.6)
    plt.title(f'{measure} Values Over Time (Window=12)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)

    # ... [rest of the plotting code remains the same]
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f'{measure}_analysis.png'), bbox_inches='tight')
    plt.close()

def main():
    factors = ['Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']
    plots_dir = os.path.join(folder_path, 'analysis_plots')
    os.makedirs(plots_dir, exist_ok=True)
    all_results = []

    for measure in factors:
        print(f"\nProcessing measure: {measure}")
        processed_data = []

        # Find Excel files
        excel_files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx") and not f.startswith("Model_Comparison")]

        # Process each file
        for filename in excel_files:
            file_path = os.path.join(folder_path, filename)
            print(f"\nProcessing file: {file_path}")

            try:
                df = pd.read_excel(file_path)
                if measure in df.columns:
                    processed_df = process_one_measure(df.copy(), measure)
                    processed_df['Source_File'] = os.path.basename(file_path).replace('.xlsx', '')
                    processed_data.append(processed_df)
                    print(f"Successfully processed {filename}")
            except Exception as e:
                print(f"Error processing {filename}: {str(e)}")

        if processed_data:
            # Combine all processed data
            combined_data = pd.concat(processed_data)
            combined_data['Date'] = pd.to_datetime(combined_data['Date'])
            combined_data.set_index('Date', inplace=True)

            # Create feature matrix
            features = pd.DataFrame({
                'Value': combined_data[measure],
                'Rolling_Mean_12': combined_data[f'{measure}_Rolling_Mean_12'],
                'Rolling_Std_12': combined_data[f'{measure}_Rolling_Std_12'],
                'Rolling_Mean_24': combined_data[f'{measure}_Rolling_Mean_24'],
                'Rolling_Std_24': combined_data[f'{measure}_Rolling_Std_24'],
                'Fund': combined_data['Source_File'],
                'Anomaly_Label_12': combined_data[f'{measure}_Dynamic_Outlier_12'].astype(int)
            })

            features = features.dropna()

            # Prepare data for modeling
            X = features[['Value', 'Rolling_Mean_12', 'Rolling_Std_12', 'Rolling_Mean_24', 'Rolling_Std_24']]
            y = features['Anomaly_Label_12']

            # Scale features
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)
            X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

            # Plot factor analysis
            plot_factor_analysis(combined_data, measure, plots_dir)

            # Initialize active learners
            active_learners = {
                'Random Forest': ActiveLearner(
                    RandomForestClassifier(n_estimators=100, random_state=42),
                    initial_size=0.1,
                    query_size=10
                ),
                'Logistic Regression': ActiveLearner(
                    LogisticRegression(max_iter=1000, random_state=42),
                    initial_size=0.1,
                    query_size=10
                ),
                'SVM': ActiveLearner(
                    SVC(probability=True, random_state=42),
                    initial_size=0.1,
                    query_size=10
                ),
                'Gradient Boosting': ActiveLearner(
                    GradientBoostingClassifier(random_state=42),
                    initial_size=0.1,
                    query_size=10
                )
            }

            # Perform active learning for each classifier
            for name, active_learner in active_learners.items():
                print(f"\nPerforming active learning for {name} on {measure}...")

                X_np = X_scaled.to_numpy()
                y_np = y.to_numpy()

                initial_score = active_learner.initialize_training(X_np, y_np)
                print(f"Initial score: {initial_score:.4f}")

                for i in range(10):  # 10 active learning iterations
                    query_indices = active_learner.query_samples(X_np)
                    score = active_learner.update(X_np, y_np, query_indices)
                    print(f"Iteration {i+1}, Score: {score:.4f}")

                plot_learning_curve(active_learner, plots_dir, f"{measure}_{name}")

                final_results = {
                    'Model': name,
                    'Measure': measure,
                    'Initial_Score': initial_score,
                    'Final_Score': score,
                    'N_Labeled_Samples': len(active_learner.labeled_indices),
                    'Learning_Rate': (score - initial_score) / 10
                }
                all_results.append(final_results)

            # Save results to Excel
            final_results_df = pd.DataFrame(all_results)
            final_results_path = os.path.join(folder_path, "Active_Learning_Results.xlsx")

            with pd.ExcelWriter(final_results_path, engine='openpyxl') as writer:
                final_results_df.to_excel(writer, sheet_name='Overall_Results', index=False)

                # Create summary sheets
                pivot_measure = final_results_df.pivot_table(
                    index='Model',
                    columns='Measure',
                    values=['Final_Score', 'Learning_Rate'],
                    aggfunc='mean'
                )
                pivot_measure.to_excel(writer, sheet_name='Summary_by_Measure')

                pivot_model = final_results_df.pivot_table(
                    index='Measure',
                    columns='Model',
                    values=['Final_Score', 'Learning_Rate'],
                    aggfunc='mean'
                )
                pivot_model.to_excel(writer, sheet_name='Summary_by_Model')

            print(f"Results saved to {final_results_path}")
        else:
            print(f"No data processed for measure: {measure}")

if __name__ == "__main__":
    main()


Processing measure: Alpha..annualisiert.

Processing file: /content/drive/My Drive/Factordata/zCapital.xlsx
Successfully processed zCapital.xlsx

Processing file: /content/drive/My Drive/Factordata/3645.xlsx
Successfully processed 3645.xlsx

Processing file: /content/drive/My Drive/Factordata/creditsuisse.xlsx
Successfully processed creditsuisse.xlsx

Processing file: /content/drive/My Drive/Factordata/21216.xlsx
Successfully processed 21216.xlsx

Processing file: /content/drive/My Drive/Factordata/IAM.xlsx
Successfully processed IAM.xlsx

Processing file: /content/drive/My Drive/Factordata/GAM.xlsx
Successfully processed GAM.xlsx

Processing file: /content/drive/My Drive/Factordata/Vontobel.xlsx
Successfully processed Vontobel.xlsx

Processing file: /content/drive/My Drive/Factordata/Lo.xlsx
Successfully processed Lo.xlsx

Processing file: /content/drive/My Drive/Factordata/SaraSelect.xlsx
Successfully processed SaraSelect.xlsx

Processing file: /content/drive/My Drive/Factordata/Fin

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import os
from datetime import datetime

class ActiveLearner:
    def __init__(self, estimator, initial_size=0.1, query_size=10):
        self.estimator = estimator
        self.initial_size = initial_size
        self.query_size = query_size
        self.labeled_indices = set()

    def initialize_training(self, X, y):
        n_initial = int(len(X) * self.initial_size)
        initial_indices = np.random.choice(len(X), n_initial, replace=False)
        self.labeled_indices.update(initial_indices)
        self.estimator.fit(X[list(self.labeled_indices)], y[list(self.labeled_indices)])
        return self.estimator.score(X[list(self.labeled_indices)], y[list(self.labeled_indices)])

    def query_samples(self, X):
        unlabeled_indices = list(set(range(len(X))) - self.labeled_indices)
        if not unlabeled_indices:
            return []

        # Get probabilities for unlabeled samples
        probs = self.estimator.predict_proba(X[unlabeled_indices])
        uncertainties = 1 - np.max(probs, axis=1)

        # Select most uncertain samples
        n_query = min(self.query_size, len(unlabeled_indices))
        query_idx = np.argsort(uncertainties)[-n_query:]
        return [unlabeled_indices[i] for i in query_idx]

    def update(self, X, y, query_indices):
        self.labeled_indices.update(query_indices)
        self.estimator.fit(X[list(self.labeled_indices)], y[list(self.labeled_indices)])
        return self.estimator.score(X[list(self.labeled_indices)], y[list(self.labeled_indices)])

class FeatureLoader:
    def __init__(self, feature_folder):
        self.feature_folder = feature_folder

    def load_unemployment(self):
        """Load unemployment data with semicolon separator"""
        path = os.path.join(self.feature_folder, 'combined_unemployment_data.csv')
        try:
            df = pd.read_csv(path, sep=';')
            print(f"Unemployment data columns: {df.columns.tolist()}")
            df['Date'] = pd.to_datetime(df['Date'])
            df['UnemploymentRate'] = df['UnemploymentRate'].astype(str).str.replace(',', '.').astype(float)
            return df[['Date', 'UnemploymentRate']]
        except Exception as e:
            print(f"Error loading unemployment data: {str(e)}")
            print(f"File path: {path}")
            raise

    def load_gdp(self):
        """Load GDP data"""
        path = os.path.join(self.feature_folder, 'gdp_data.csv')
        try:
            df = pd.read_csv(path)
            print(f"GDP data columns: {df.columns.tolist()}")
            df['Date'] = pd.to_datetime(df['year_quarter'])
            return df[['Date', 'tsd_cleaned', 'real_q']]
        except Exception as e:
            print(f"Error loading GDP data: {str(e)}")
            print(f"File path: {path}")
            raise

    def load_ks(self):
        """Load KS data"""
        path = os.path.join(self.feature_folder, 'ks_q_hist.csv')
        try:
            df = pd.read_csv(path)
            print(f"KS data columns: {df.columns.tolist()}")
            df['Date'] = pd.to_datetime(df['date'])
            return df[['Date', 'value']]
        except Exception as e:
            print(f"Error loading KS data: {str(e)}")
            print(f"File path: {path}")
            raise

    def load_inflation(self):
        """Load inflation data"""
        path = os.path.join(self.feature_folder, 'inflation.csv')
        try:
            df = pd.read_csv(path)
            print(f"Inflation data columns: {df.columns.tolist()}")
            df['Date'] = pd.to_datetime(df['Date'])
            for col in ['chinf', 'usinf', 'deinf']:
                df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.').replace('NA', np.nan),
                                      errors='coerce')
            return df[['Date', 'chinf', 'usinf', 'deinf']]
        except Exception as e:
            print(f"Error loading inflation data: {str(e)}")
            print(f"File path: {path}")
            raise

    def load_currency_rates(self):
        """Load EUR/CHF, USD/EUR, and USD/CHF monthly rates"""
        try:
            # Load EUR/CHF data
            eur_chf_path = os.path.join(self.feature_folder, 'EUR_CHF_monthly_rates.csv')
            eur_chf_df = pd.read_csv(eur_chf_path)
            print(f"EUR/CHF data columns: {eur_chf_df.columns.tolist()}")
            eur_chf_df['Date'] = pd.to_datetime(eur_chf_df['Date'])
            eur_chf_df = eur_chf_df.rename(columns={'Rate': 'EUR_CHF_rate'})

            # Load USD/EUR data
            usd_eur_path = os.path.join(self.feature_folder, 'USD_EUR_monthly_rates.csv')
            usd_eur_df = pd.read_csv(usd_eur_path)
            print(f"USD/EUR data columns: {usd_eur_df.columns.tolist()}")
            usd_eur_df['Date'] = pd.to_datetime(usd_eur_df['Date'])
            usd_eur_df = usd_eur_df.rename(columns={'Rate': 'USD_EUR_rate'})

            # Load USD/CHF data
            usd_chf_path = os.path.join(self.feature_folder, 'USD_CHF_monthly_rates.csv')
            usd_chf_df = pd.read_csv(usd_chf_path)
            print(f"USD/CHF data columns: {usd_chf_df.columns.tolist()}")
            usd_chf_df['Date'] = pd.to_datetime(usd_chf_df['Date'])
            usd_chf_df = usd_chf_df.rename(columns={'Rate': 'USD_CHF_rate'})

            # Merge all currency data
            currencies = pd.merge_asof(
                eur_chf_df.sort_values('Date'),
                usd_eur_df.sort_values('Date'),
                on='Date',
                direction='nearest'
            )
            currencies = pd.merge_asof(
                currencies.sort_values('Date'),
                usd_chf_df.sort_values('Date'),
                on='Date',
                direction='nearest'
            )

            return currencies

        except Exception as e:
            print(f"Error loading currency data: {str(e)}")
            print("Current directory:", os.getcwd())
            raise

    def load_all_features(self):
        print("Loading external features...")
        unemployment = self.load_unemployment()
        gdp = self.load_gdp()
        ks = self.load_ks()
        inflation = self.load_inflation()
        currencies = self.load_currency_rates()
        return unemployment, gdp, ks, inflation, currencies

class DataProcessor:
    @staticmethod
    def process_measure(df, measure):
        """Add rolling statistics for both window sizes."""
        for window in [12, 24]:
            df[f'{measure}_Rolling_Mean_{window}'] = df[measure].rolling(window=window, min_periods=1).mean()
            df[f'{measure}_Rolling_Std_{window}'] = df[measure].rolling(window=window, min_periods=1).std()
            df[f'{measure}_Dynamic_Outlier_{window}'] = (
                (df[measure] - df[f'{measure}_Rolling_Mean_{window}']).abs() >
                2 * df[f'{measure}_Rolling_Std_{window}']
            )
        return df

    @staticmethod
    def merge_features(base_df, unemployment, gdp, ks, inflation, currencies):
        """Merge all feature datasets with the base dataframe."""
        base_df = base_df.reset_index()
        base_df['Date'] = pd.to_datetime(base_df['Date'])

        # Sequential merging using merge_asof
        df = pd.merge_asof(base_df.sort_values('Date'),
                          unemployment.sort_values('Date'),
                          on='Date',
                          direction='nearest')

        df = pd.merge_asof(df.sort_values('Date'),
                          gdp.sort_values('Date'),
                          on='Date',
                          direction='nearest')

        df = pd.merge_asof(df.sort_values('Date'),
                          ks[['Date', 'value']].sort_values('Date'),
                          on='Date',
                          direction='nearest',
                          suffixes=('', '_ks'))

        df = pd.merge_asof(df.sort_values('Date'),
                          inflation.sort_values('Date'),
                          on='Date',
                          direction='nearest')

        df = pd.merge_asof(df.sort_values('Date'),
                          currencies.sort_values('Date'),
                          on='Date',
                          direction='nearest')

        return df.set_index('Date')

class ModelTrainer:
    def __init__(self):
        self.active_learners = {
            'Random Forest': ActiveLearner(
                RandomForestClassifier(n_estimators=100, random_state=42),
                initial_size=0.1,
                query_size=10
            ),
            'Logistic Regression': ActiveLearner(
                LogisticRegression(max_iter=1000, random_state=42),
                initial_size=0.1,
                query_size=10
            ),
            'SVM': ActiveLearner(
                SVC(probability=True, random_state=42),
                initial_size=0.1,
                query_size=10
            ),
            'Gradient Boosting': ActiveLearner(
                GradientBoostingClassifier(random_state=42),
                initial_size=0.1,
                query_size=10
            )
        }

    def train_models(self, X, y):
        results = []
        unique_classes = np.unique(y)

        for name, learner in self.active_learners.items():
            print(f"\nTraining {name}...")
            X_np = X.to_numpy()
            y_np = y.to_numpy()

            if len(unique_classes) < 2:
                print(f"Warning: Only one class present ({unique_classes[0]}). Skipping {name}.")
                results.append({
                    'Model': name,
                    'Initial_Score': 1.0,  # perfect score for single class
                    'Final_Score': 1.0,
                    'N_Labeled_Samples': len(y),
                    'Learning_Rate': 0.0,
                    'Feature_Importance': None,
                    'Note': 'Single class data'
                })
                continue

            try:
                initial_score = learner.initialize_training(X_np, y_np)
                print(f"Initial score: {initial_score:.4f}")

                for i in range(10):
                    query_indices = learner.query_samples(X_np)
                    if not query_indices:
                        break
                    score = learner.update(X_np, y_np, query_indices)
                    print(f"Iteration {i+1}, Score: {score:.4f}")

                results.append({
                    'Model': name,
                    'Initial_Score': initial_score,
                    'Final_Score': score,
                    'N_Labeled_Samples': len(learner.labeled_indices),
                    'Learning_Rate': (score - initial_score) / 10,
                    'Feature_Importance': self.get_feature_importance(learner, X.columns)
                })
            except Exception as e:
                print(f"Error training {name}: {str(e)}")
                results.append({
                    'Model': name,
                    'Initial_Score': None,
                    'Final_Score': None,
                    'N_Labeled_Samples': 0,
                    'Learning_Rate': None,
                    'Feature_Importance': None,
                    'Error': str(e)
                })

        return results

    @staticmethod
    def get_feature_importance(learner, feature_names):
        model = learner.estimator
        if hasattr(model, 'feature_importances_'):
            return dict(zip(feature_names, model.feature_importances_))
        elif hasattr(model, 'coef_'):
            return dict(zip(feature_names, abs(model.coef_[0])))
        return None

class ResultsSaver:
    def __init__(self, save_path):
        self.save_path = save_path

    def save_results(self, results, measure):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"Active_Learning_Results_{measure}_{timestamp}.xlsx"
        final_results_path = os.path.join(self.save_path, filename)

        # Create main results dataframe
        results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'Feature_Importance'}
                                 for r in results])

        with pd.ExcelWriter(final_results_path, engine='openpyxl') as writer:
            results_df.to_excel(writer, sheet_name='Overall_Results', index=False)

            # Feature importance sheet
            feature_importance_data = []
            for result in results:
                if result.get('Feature_Importance'):
                    for feature, importance in result['Feature_Importance'].items():
                        feature_importance_data.append({
                            'Model': result['Model'],
                            'Feature': feature,
                            'Importance': importance
                        })

            if feature_importance_data:
                pd.DataFrame(feature_importance_data).to_excel(
                    writer,
                    sheet_name='Feature_Importance',
                    index=False
                )

        print(f"Results saved to {final_results_path}")

def main():
    # Paths setup
    factor_folder = '/content/drive/My Drive/Factordata'
    feature_folder = '/content/drive/My Drive/feature'
    plots_dir = os.path.join(factor_folder, 'analysis_plots')
    os.makedirs(plots_dir, exist_ok=True)

    # Initialize components
    feature_loader = FeatureLoader(feature_folder)
    data_processor = DataProcessor()
    model_trainer = ModelTrainer()
    results_saver = ResultsSaver(factor_folder)

    # Load external features
    unemployment, gdp, ks, inflation, exchange_rate = feature_loader.load_all_features()

    # Process each factor
    factors = ['Alpha..annualisiert.', 'Value.Growth', 'Small.Large', 'Momentum', 'Volatility']
    for measure in factors:
        print(f"\nProcessing measure: {measure}")
        processed_data = []

        # Process Excel files
        excel_files = [f for f in os.listdir(factor_folder)
                      if f.endswith(".xlsx") and not f.startswith("Model_Comparison")]

        for filename in excel_files:
            file_path = os.path.join(factor_folder, filename)
            try:
                df = pd.read_excel(file_path)
                if measure in df.columns:
                    processed_df = data_processor.process_measure(df.copy(), measure)
                    processed_df['Source_File'] = os.path.basename(file_path).replace('.xlsx', '')
                    processed_data.append(processed_df)
                    print(f"Successfully processed {filename}")
            except Exception as e:
                print(f"Error processing {filename}: {str(e)}")

        if processed_data:
            # Combine and process data
            combined_data = pd.concat(processed_data)
            combined_data['Date'] = pd.to_datetime(combined_data['Date'])
            combined_data.set_index('Date', inplace=True)

            # Merge with external features
            enhanced_data = data_processor.merge_features(
                combined_data, unemployment, gdp, ks, inflation, exchange_rate
            )

            # Prepare features for modeling
            features = pd.DataFrame({
                'Value': enhanced_data[measure],
                'Rolling_Mean_12': enhanced_data[f'{measure}_Rolling_Mean_12'],
                'Rolling_Std_12': enhanced_data[f'{measure}_Rolling_Std_12'],
                'Rolling_Mean_24': enhanced_data[f'{measure}_Rolling_Mean_24'],
                'Rolling_Std_24': enhanced_data[f'{measure}_Rolling_Std_24'],
                'UnemploymentRate': enhanced_data['UnemploymentRate'],
                'GDP_Real': enhanced_data['real_q'],
                'KS_Value': enhanced_data['value'],
                'US_Inflation': enhanced_data['usinf'],
                'China_Inflation': enhanced_data['chinf'],
                'German_Inflation': enhanced_data['deinf'],
                'EUR_CHF_Rate': enhanced_data['EUR_CHF_rate'],
                'USD_EUR_Rate': enhanced_data['USD_EUR_rate'],
                'USD_CHF_Rate': enhanced_data['USD_CHF_rate']
            }).dropna()

            # Prepare target variable
            target = enhanced_data[f'{measure}_Dynamic_Outlier_12'].astype(int)
            target = target[features.index]

            # Scale features
            scaler = StandardScaler()
            features_scaled = pd.DataFrame(
                scaler.fit_transform(features),
                columns=features.columns,
                index=features.index
            )

            # Train models and save results
            results = model_trainer.train_models(features_scaled, target)
            results_saver.save_results(results, measure)
        else:
            print(f"No data processed for measure: {measure}")

if __name__ == "__main__":
    main()

Loading external features...
Unemployment data columns: ['Date', 'UnemploymentRate']
GDP data columns: ['year_quarter', 'tsd_cleaned', 'real_q']
KS data columns: ['Unnamed: 0', 'structure', 'type', 'seas_adj', 'date', 'value']
Inflation data columns: ['Date', 'chinf', 'usinf', 'deinf']
EUR/CHF data columns: ['Date', 'Rate']
USD/EUR data columns: ['Date', 'Rate']
USD/CHF data columns: ['Date', 'Rate']

Processing measure: Alpha..annualisiert.
Successfully processed zCapital.xlsx
Successfully processed 3645.xlsx
Successfully processed creditsuisse.xlsx
Successfully processed 21216.xlsx
Successfully processed IAM.xlsx
Successfully processed GAM.xlsx
Successfully processed Vontobel.xlsx
Successfully processed Lo.xlsx
Successfully processed SaraSelect.xlsx
Successfully processed Finreon.xlsx
Successfully processed SGKB.xlsx
Successfully processed Pictet.xlsx

Training Random Forest...
Initial score: 1.0000
Iteration 1, Score: 1.0000
Iteration 2, Score: 1.0000
Iteration 3, Score: 1.0000
Iter